# CV-8 — preparação e treinamento controlado no Colab

Tech Challenge — Fase 1, FIAP. MobileNetV3 Small com pesos ImageNet obrigatórios.
**Este notebook foi entregue sem execução. Nenhum treinamento foi realizado.**

Execute em ordem para revisar dependências, fontes, Drive, dados, pesos e
configuração. Há dois controles independentes, inicialmente desabilitados:
`ALLOW_WEIGHT_DOWNLOAD=False` e `RUN_TRAINING=False`. O primeiro autoriza apenas
obter os pesos oficiais; o segundo autoriza épocas depois da revisão.

O teste oficial permanece reservado. Só os manifestos train/val e suas imagens
são utilizados. Leia `docs/vision_training.md` antes da primeira execução.

## 1. Ambiente, versões e disponibilidade real de GPU

A inspeção é a da CV-6, acrescida de matplotlib para as curvas. Não instala nada.
CPU é permitida; uma GPU não é presumida. A configuração posterior escolhe auto,
cpu ou cuda. A preparação de pesos pode usar o Drive mesmo sem GPU.

In [ ]:
import importlib.metadata as metadata
import json
import platform
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
print("Sistema:", platform.platform())
for package in ("torch", "torchvision", "numpy", "pandas", "Pillow", "scikit-learn", "matplotlib"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "ausente")

PROBE_SOURCE = r"""
import importlib, json
info = {"torch_compatible": False, "dependencies": {}, "errors": {}}
try:
    import torch, torchvision
    torchvision.ops.nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5)
    info.update(torch_compatible=True, torch=torch.__version__, torchvision=torchvision.__version__,
                cuda_build=torch.version.cuda, cuda_available=torch.cuda.is_available(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except Exception as exc:
    info["errors"]["torch_stack"] = repr(exc)
for module in ("numpy", "pandas", "PIL", "sklearn", "matplotlib"):
    try:
        loaded = importlib.import_module(module)
        info["dependencies"][module] = getattr(loaded, "__version__", "importado")
    except Exception as exc:
        info["errors"][module] = repr(exc)
info["ok"] = info["torch_compatible"] and len(info["dependencies"]) == 5
print(json.dumps(info))
"""

def probe_environment():
    probe = subprocess.run([sys.executable, "-c", PROBE_SOURCE], capture_output=True,
                           text=True, timeout=120)
    if probe.returncode:
        raise RuntimeError("Falha na inspeção: " + probe.stderr)
    if probe.stderr.strip():
        print("Avisos da inspeção:", probe.stderr.strip())
    return json.loads(probe.stdout.strip().splitlines()[-1])

environment = probe_environment()
print(json.dumps(environment, indent=2))

## 2. Código CV-8 atualizado

Use o ZIP **tech-challenge-cv8-code.zip**, não o ZIP CV-6 antigo, que não contém o
treinamento. Alternativamente, após publicação, selecione github e preencha a
branch. Nenhuma publicação automática é realizada. A extração de código é feita
em pasta temporária nova, sem dados, pesos ou ambientes virtuais.

In [ ]:
import hashlib
import io
import os
import stat
import tempfile
import zipfile

CODE_SOURCE = "upload_zip"  # upload_zip, github ou existing
GITHUB_URL = "https://github.com/Caio-crl/breast-cancer-ml-classification.git"
GITHUB_BRANCH = ""  # preencher somente depois de publicar a branch de Computer Vision
EXISTING_ROOT = Path("/content/tech-challenge-cv")
CODE_FILES = {
    "src/__init__.py", "src/vision/__init__.py", "src/vision/inspect_dataset.py",
    "src/vision/prepare_manifest.py", "src/vision/dataset.py", "src/vision/model.py",
    "src/vision/smoke_test.py", "tests/vision/test_inspect_dataset.py",
    "tests/vision/test_prepare_manifest.py", "tests/vision/test_dataset.py",
    "tests/vision/test_model.py", "requirements-vision.txt", "docs/vision_infrastructure.md",
}

CODE_FILES.update({"src/vision/training.py", "tests/vision/test_training.py", "docs/vision_training.md"})

if CODE_SOURCE == "upload_zip":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("Envie somente o ZIP de código.")
    payload = next(iter(uploaded.values()))
    print("SHA-256 do ZIP:", hashlib.sha256(payload).hexdigest())
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="tech-challenge-cv8-code-"))
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = [entry for entry in archive.infolist() if not entry.is_dir()]
        names = [entry.filename for entry in entries]
        if len(names) != len(set(names)) or set(names) != CODE_FILES:
            raise ValueError("ZIP diferente da lista de código esperada; não inclua dados ou ambientes.")
        if sum(entry.file_size for entry in entries) > 5_000_000:
            raise ValueError("ZIP de código maior que o limite esperado.")
        for entry in entries:
            if stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Links simbólicos não são permitidos no ZIP.")
            destination = PROJECT_ROOT / entry.filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(entry))
    del uploaded, payload
elif CODE_SOURCE == "github":
    if not GITHUB_BRANCH.strip() or GITHUB_BRANCH.startswith("-"):
        raise ValueError("Informe a branch publicada; nenhum nome foi presumido.")
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="tech-challenge-cv8-git-"))
    subprocess.run(["git", "clone", "--single-branch", "--branch", GITHUB_BRANCH,
                    GITHUB_URL, str(PROJECT_ROOT)], check=True)
    print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"],
                                             cwd=PROJECT_ROOT, text=True).strip())
elif CODE_SOURCE == "existing":
    PROJECT_ROOT = EXISTING_ROOT.expanduser().resolve()
else:
    raise ValueError("CODE_SOURCE inválido.")

missing = sorted(name for name in CODE_FILES if not (PROJECT_ROOT / name).is_file())
if missing:
    raise FileNotFoundError("Código CV-8 incompleto: " + repr(missing))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Raiz do código:", PROJECT_ROOT)
source_hashes = {name: hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest()
                 for name in sorted(CODE_FILES)}
print("Identificação dos fontes:", json.dumps(source_hashes, indent=2))

## 3. Dependências: nenhuma reinstalação se estiverem funcionais

Mantenha INSTALL_IF_NEEDED=False. Se a inspeção identificar erro, habilite a
instalação apenas após revisão; o procedimento é o da CV-6, com par de referência
torch 2.7.1 / torchvision 0.22.1 e wheels CPU como fallback explícito. A pilha GPU
funcional já fornecida pelo Colab é preservada. Reinicie a sessão se instalar.

In [ ]:
INSTALL_IF_NEEDED = False
USE_CPU_WHEELS = True  # suficiente para testes e forward; GPU pode usar a pilha funcional já fornecida
environment = probe_environment()
if not environment["ok"]:
    if not INSTALL_IF_NEEDED:
        raise RuntimeError("Dependências incompatíveis/ausentes. Revise os erros acima; "
                           "habilite INSTALL_IF_NEEDED apenas se necessário no Colab.")
    if "google.colab" not in sys.modules:
        import importlib.util
        if importlib.util.find_spec("google.colab") is None:
            raise RuntimeError("Instalação opcional restrita ao Google Colab.")
    auxiliary = {"numpy": "numpy==1.26.4", "pandas": "pandas==2.2.3",
                 "PIL": "Pillow>=11,<14", "sklearn": "scikit-learn==1.6.1", "matplotlib": "matplotlib==3.10.1"}
    needed = [requirement for module, requirement in auxiliary.items()
              if module not in environment["dependencies"]]
    if needed:
        subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", *needed], check=True)
    if not environment["torch_compatible"]:
        command = [sys.executable, "-m", "pip", "install", "--force-reinstall", "torch==2.7.1", "torchvision==0.22.1"]
        if USE_CPU_WHEELS:
            command += ["--index-url", "https://download.pytorch.org/whl/cpu"]
        subprocess.run(command, check=True)
    raise RuntimeError("Instalação solicitada concluída. Reinicie a sessão antes de continuar.")
print("Ambiente funcional; nenhuma reinstalação necessária.")

## 4. Testes offline antes de qualquer treinamento real

Os testes novos usam CPU e fixtures sintéticas temporárias. Verificam métricas,
uma atualização pequena, features congeladas, seleção, early stopping, carga de
checkpoints e exclusão do teste. Não usam dados reais nem pesos baixados. Devem
passar junto aos testes anteriores, sem falhas ou skips. Os testes do loop não
equivalem a executar épocas do experimento CBIS-DDSM.

In [ ]:
import unittest
import pandas as pd

class RecordingResult(unittest.TextTestResult):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statuses = {}
    def addSuccess(self, test):
        super().addSuccess(test)
        self.statuses[test.id()] = "aprovado"
    def addSkip(self, test, reason):
        super().addSkip(test, reason)
        self.statuses[test.id()] = "ignorado: " + reason
    def addFailure(self, test, err):
        super().addFailure(test, err)
        self.statuses[test.id()] = "falha"
    def addError(self, test, err):
        super().addError(test, err)
        self.statuses[test.id()] = "erro"

suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / "tests/vision"), pattern="test_*.py")
test_result = unittest.TextTestRunner(verbosity=2, resultclass=RecordingResult).run(suite)
torch_runtime_tests = pd.DataFrame([
    {"teste": name, "resultado": status}
    for name, status in sorted(test_result.statuses.items())
    if name.startswith(("test_dataset.DatasetTests.", "test_model.ModelTests.", "test_training.TrainingRuntimeTests."))
])
display(torch_runtime_tests)
test_summary = {"discovered": test_result.testsRun,
                "approved": sum(status == "aprovado" for status in test_result.statuses.values()),
                "skipped": len(test_result.skipped), "failures": len(test_result.failures),
                "errors": len(test_result.errors)}
print(test_summary)
assert test_result.wasSuccessful() and not test_result.skipped, "Há falhas ou skips pendentes."
assert len(torch_runtime_tests) == 31, "Os 31 testes dependentes de torch esperados não foram identificados; confira a versão do código."
assert torch_runtime_tests.resultado.eq("aprovado").all(), "Nem todos os testes dependentes de torch foram aprovados."

## 5. Drive e pacote CV-7 — somente treino/validação

Monte o Drive e ajuste os caminhos; não se presume que o ZIP já tenha sido
enviado. Caso DATA_BASE já contenha os dois manifestos e a raiz das imagens, eles
serão reutilizados sem alteração. Caso contrário, o ZIP CV-7 precisa existir.

A extração seleciona apenas os manifestos train/val e os JPGs que referenciam,
verificando SHA-256 incrementalmente. **Não extrai nem lê o manifesto ou as
imagens de teste.** Os originais no pacote são preservados para avaliação futura.
Não mescla nem sobrescreve uma pasta existente incompleta.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PACKAGE = Path("/content/drive/MyDrive/tech-challenge/cbis_ddsm_cv7.zip")
DATA_BASE = Path("/content/cv8_data")
DATASET_ROOT = DATA_BASE / "cbis_ddsm"
MANIFEST_DIR = DATA_BASE / "processed"
WEIGHTS_PATH = Path("/content/drive/MyDrive/tech-challenge/weights/mobilenet_v3_small-047dcff4.pth")
OUTPUT_DIR = Path("/content/drive/MyDrive/tech-challenge/artifacts/vision/run_001")

def ensure_training_data(package, destination):
    from pathlib import PurePosixPath
    import shutil
    manifest_names = [f"processed/manifest_{s}.csv" for s in ("train", "val")]
    if destination.exists():
        if not (destination / "cbis_ddsm").is_dir() or not all((destination / name).is_file() for name in manifest_names):
            raise FileExistsError("Destino existente incompleto. Escolha outro DATA_BASE; nada foi sobrescrito.")
        print("Cópia existente reutilizada; a preparação validará somente train/val.")
        return
    if not package.is_file():
        raise FileNotFoundError(f"Envie o pacote CV-7 ao Drive e ajuste DATA_PACKAGE: {package}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Entradas duplicadas no ZIP de dados.")
        snapshots = {name: archive.read(name) for name in manifest_names}
        references = {}
        for name, payload in snapshots.items():
            frame = pd.read_csv(io.BytesIO(payload), dtype=str, keep_default_na=False)
            expected_split = "train" if name.endswith("manifest_train.csv") else "val"
            if frame.empty or not frame.split.eq(expected_split).all():
                raise ValueError("Partição do manifesto inconsistente.")
            for row in frame.itertuples(index=False):
                relative = PurePosixPath(row.image_path)
                if (relative.is_absolute() or ".." in relative.parts or "\\" in row.image_path
                        or relative.as_posix() != row.image_path or len(relative.parts) != 3
                        or relative.parts[0] != "jpeg" or relative.suffix.lower() != ".jpg"):
                    raise ValueError("Caminho não portátil no manifesto.")
                member = "cbis_ddsm/" + relative.as_posix()
                if member in references or member not in names:
                    raise ValueError("Imagem repetida ou ausente no pacote.")
                references[member] = row.sha256
        with tempfile.TemporaryDirectory(prefix="cv8-extract-", dir=destination.parent) as staging:
            staged = Path(staging) / "data"
            for name, payload in snapshots.items():
                target = staged / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(payload)  # bytes originais, sem reescrever o CSV
            for index, (name, expected) in enumerate(references.items(), 1):
                target = staged / name
                target.parent.mkdir(parents=True, exist_ok=True)
                digest = hashlib.sha256()
                with archive.open(name) as source, target.open("xb") as output:
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        digest.update(chunk)
                        output.write(chunk)
                if digest.hexdigest() != expected:
                    raise ValueError(f"Hash divergente: {name}")
                if index % 200 == 0:
                    print("Recortes train/val extraídos:", index)
            if destination.exists():
                raise FileExistsError("Destino passou a existir; nada será sobrescrito.")
            staged.rename(destination)
        print("Extração validada:", len(references), "JPGs; somente treino/validação.")

ensure_training_data(DATA_PACKAGE, DATA_BASE)

## 6. Configuração inicial para revisão

Cabeça classificadora apenas, batch 8, AdamW lr=0,001 e weight decay=0,0001.
Seed 42, máximo 10 épocas, early stopping patience=3/min_delta=0,0001 baseado
no F1 de validação. OUTPUT_DIR deve ser uma pasta nova; alterar parâmetros exige
preparar uma nova sessão. Não há otimização com resultados de teste.

In [ ]:
from dataclasses import asdict
from src.vision.training import TrainingConfig, prepare_training, fit, verify_weight_file

config = TrainingConfig(
    dataset_root=str(DATASET_ROOT), manifest_dir=str(MANIFEST_DIR), output_dir=str(OUTPUT_DIR),
    weights_path=str(WEIGHTS_PATH), seed=42, batch_size=8, num_workers=0,
    learning_rate=1e-3, weight_decay=1e-4, max_epochs=10,
    patience=3, min_delta=1e-4, device="auto", trainable_feature_blocks=0,
)
config.validate()
print(json.dumps(asdict(config), indent=2))
print("Seleção: maior F1 de validação com melhora > min_delta; empates mantêm a época anterior.")
print("Somente treino e validação; teste reservado.")

## 7. Pesos ImageNet obrigatórios — download separado e explícito

Sem arquivo válido, esta célula interrompe a preparação. Só altere
ALLOW_WEIGHT_DOWNLOAD para True após autorizar a obtenção dos pesos oficiais.
Isso **não** autoriza épocas. A URL vem do enum do torchvision; verifica-se o
prefixo SHA-256 oficial e registra-se o hash completo. Não há sobrescrita de
arquivo existente ou fallback para pesos aleatórios. A obtenção ocorre somente
neste runtime Colab, nunca automaticamente nos testes.

In [ ]:
import torch
from torchvision.models import MobileNet_V3_Small_Weights

ALLOW_WEIGHT_DOWNLOAD = False
weights_url = MobileNet_V3_Small_Weights.IMAGENET1K_V1.url
if not WEIGHTS_PATH.is_file():
    if not ALLOW_WEIGHT_DOWNLOAD:
        raise FileNotFoundError("Pesos ImageNet ausentes. Disponibilize o arquivo oficial ou autorize explicitamente esta célula.")
    import re
    from urllib.parse import urlparse
    prefix = re.search(r"-([0-9a-f]{8,64})\.pth$", urlparse(weights_url).path)[1]
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="cv8-weights-", dir=WEIGHTS_PATH.parent) as temporary:
        candidate = Path(temporary) / "weights.pth"
        torch.hub.download_url_to_file(weights_url, str(candidate), hash_prefix=prefix, progress=True)
        verify_weight_file(candidate, weights_url)
        if WEIGHTS_PATH.exists():
            raise FileExistsError("O arquivo de pesos passou a existir; não será sobrescrito.")
        candidate.rename(WEIGHTS_PATH)
print(json.dumps(verify_weight_file(WEIGHTS_PATH, weights_url), indent=2))

## 8. Preparar modelo e loaders — nenhuma época

Valida hash, carga estrita e igualdade dos tensores pré-treinados. Usa somente
manifest_train e manifest_val. Não executa forward/backward nem cria OUTPUT_DIR.
Confira contagens, parâmetros treináveis, dispositivo, pesos e configuração antes
de prosseguir. Features congeladas aleatórias não são aceitas.

In [ ]:
prepared = prepare_training(config)
assert set(prepared.loaders) == {"train", "val"}
assert prepared.metadata["pretrained_weights"]["backbone_load_verified"]
assert prepared.metadata["pretrained_weights"]["hash_verified"]
assert not any(p.requires_grad for p in prepared.model.features.parameters())
assert not Path(config.output_dir).exists()
print(json.dumps(prepared.metadata, indent=2))

## 9. Treinamento manual — desabilitado por padrão

Somente após revisar a célula anterior e autorizar a primeira execução controlada,
altere RUN_TRAINING para True. Os resultados serão gravados na pasta nova do Drive.
O melhor estado será restaurado ao final. Não há avaliação do teste.

In [ ]:
RUN_TRAINING = False
training_result = None
if RUN_TRAINING:
    training_result = fit(prepared)
    print(json.dumps({k: v for k, v in training_result.items() if k != "history"}, indent=2))
else:
    print("Nenhuma época executada. Revise a configuração antes de habilitar RUN_TRAINING.")

## 10. Histórico e curvas de treino/validação

Somente após uma execução manual. Loss, Accuracy, Precision, Recall e F1 referem-se
à classe maligna quando binárias. A linha vertical marca a época selecionada.
O gráfico não contém dados de teste nem demonstra generalização clínica.

In [ ]:
if training_result is None:
    print("Histórico indisponível: treinamento ainda não executado nesta sessão.")
else:
    import matplotlib.pyplot as plt
    history = pd.DataFrame(training_result["history"])
    display(history)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for axis, metric in zip(axes.flat, ("loss", "accuracy", "precision", "recall", "f1")):
        axis.plot(history.epoch, history[f"train_{metric}"], label="train")
        axis.plot(history.epoch, history[f"val_{metric}"], label="val")
        axis.axvline(training_result["best_epoch"], color="gray", linestyle="--", label="época selecionada")
        axis.set(title=metric, xlabel="Época")
        axis.legend()
    axes.flat[-1].axis("off")
    fig.tight_layout()
    plt.show()
    print("Artefatos no Drive:", training_result["output_dir"])
    print("Teste oficial permanece reservado para avaliação independente futura.")